# Adding Tools to your Agent

LLMs are smart, but they are frozen in time or lack access to your specific systems. Tools (also known as Function Calling) solve this.

In Pydantic AI, there are two main ways to give an agent a tool:
1. `@agent.tool_plain` for simple functions.
2. `@agent.tool` for functions that need securely injected data (like API keys or Database objects).

In [1]:
import nest_asyncio
nest_asyncio.apply()

import random
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv
from dataclasses import dataclass

load_dotenv()

True

### 1. Simple Tools (`@agent.tool_plain`)

Let's create an agent that can roll dice. 
**Important:** The docstring and type hints are how the AI knows WHAT the tool does!

In [3]:
game_agent = Agent(
    'groq:llama-3.3-70b-versatile',
    system_prompt="You are a helpful game master assistant."
)

@game_agent.tool_plain
def roll_dice(sides: int) -> int:
    """
    Use this to roll a dice with a specific number of sides.
    """
    print(f"\n[System] Tool Used! Rolling a {sides}-sided dice...")
    return random.randint(1, sides)

# Let's test it
response1 = game_agent.run_sync("I am attacking the dragon! Can you roll a 20-sided dice for me?")
print("\nAgent:", response1.output)


[System] Tool Used! Rolling a 20-sided dice...

Agent: The roll resulted in a 2. Unfortunately, it looks like your attack on the dragon didn't go as planned. The dragon is now ready to counterattack. Would you like to roll for defense?


### 2. Contextual Tools (`@agent.tool`)

If your tool needs to connect to a Database, you shouldn't let the LLM guess the database credentials! 
Instead, you inject it using `deps_type` and access it via `RunContext`.

In [4]:
# 1. Define what secret data we want to pass
@dataclass
class CustomerDatabase:
    company_name: str
    secret_discounts: dict

# 2. Tell the Agent to expect this type of data
store_agent = Agent(
    'groq:llama-3.3-70b-versatile',
    deps_type=CustomerDatabase,
    system_prompt="You are a store assistant. Check for discounts if asked."
)

# 3. Access our injected data through 'ctx'
@store_agent.tool
def check_discounts(ctx: RunContext[CustomerDatabase], item_name: str) -> str:
    """Use this to check if a specific item has a discount available."""
    print(f"\n[System] Checking the {ctx.deps.company_name} DB for {item_name}...")
    
    # Note we use ctx.deps to access our secure dictionary
    discount = ctx.deps.secret_discounts.get(item_name.lower(), "0%")
    return f"The discount is {discount}."

# 4. Provide the secure data at runtime!
my_live_db = CustomerDatabase(
    company_name="Tech Mega Store",
    secret_discounts={"laptop": "20%", "mouse": "50%"}
)

response2 = store_agent.run_sync("Do you have any discounts on a laptop?", deps=my_live_db)
print("\nAgent:", response2.output)


[System] Checking the Tech Mega Store DB for laptop...

Agent: We have a 20% discount on laptops. Would you like to know more about our laptop options?
